In [ ]:
!pip install bert-score
!pip install datasets
!pip install sentence-transformers datasets accelerate transformers wandb
!pip install sentence-transformers torch sklearn bert-score
!pip install sentencepiece
!pip install -q sentencepiece transformers sentence-transformers bert-score torch scikit-learn pandas numpy
!pip install -q pytorch-crf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
# @title
# ============================================================
# paraphrase-multilingual-MiniLM-L12-v2 + SPM(.model) → Unigram (tokenizers)
# MLM адаптациясы жоқ, held-out бағалау (эксперттік)
# ============================================================

# ENV
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["WANDB_DISABLED"] = "true"  # sentence-transformers → W&B өшіру

# !pip install -q -U "transformers>=4.41.0" "tokenizers>=0.15.2" \
#                   "sentence-transformers>=2.7.0" "sentencepiece>=0.1.99" "bert-score"

import time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

import sentencepiece as spm
# from bert_score import score as bertscore   # автоматты метрика қолданбаймыз

from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.processors import TemplateProcessing
from tokenizers.pre_tokenizers import Metaspace as PreMS
from tokenizers.decoders import Metaspace as DecMS

from transformers import (
    PreTrainedTokenizerFast,
    set_seed,
)

from sentence_transformers import SentenceTransformer, InputExample, losses, models
from google.colab import drive
from sklearn.model_selection import train_test_split

# --------------------------
# 🔒 Seed (strict deterministic емес, бірақ тұрақты)
# --------------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

try:
    torch.use_deterministic_algorithms(False)
except Exception:
    pass

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("🖥 DEVICE:", DEVICE)

# =========================
# 0) QA деректері – JSON файлдан оқу
# =========================
# Файл форматы: әр жолда {"question": "...", "answer": "..."} объекті
DATA_JSON_PATH = "qa_data.json"  # керек болса /content/drive/... деп толық жолға ауыстырыңыз

if not Path(DATA_JSON_PATH).is_file():
    raise FileNotFoundError(f"QA JSON файлы табылмады: {DATA_JSON_PATH}")

print(f"📥 QA JSON файлдан жүктеу: {DATA_JSON_PATH}")

import json
data = []
with open(DATA_JSON_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        # Соңында үтір тұрған болса, алып тастаймыз: {...},
        if line.endswith(","):
            line = line[:-1].strip()

        # Алдымен тікелей parse жасап көреміз
        try:
            obj = json.loads(line)
        except json.JSONDecodeError as e:
            # Егер Invalid \escape қатесі болса — backslash-тарды escape жасаймыз
            if "Invalid \\escape" in str(e):
                fixed_line = line.replace("\\", "\\\\")
                obj = json.loads(fixed_line)
            else:
                raise
        data.append(obj)

# Күтілетін формат: [{"question": "...", "answer": "..."}, ...]
qa_data = pd.DataFrame(data)
assert {"question", "answer"}.issubset(qa_data.columns), "JSON-да 'question' және 'answer' бағандары болуы тиіс."

print(f"📚 Барлық QA жазба саны: {len(qa_data)}")

# --------------------------
# TRAIN / TEST SPLIT (held-out)
# --------------------------
train_df, test_df = train_test_split(
    qa_data,
    test_size=0.1,      # ~10% TEST – тек эксперттік бағалау үшін
    random_state=SEED,
    shuffle=True,
)

train_questions = train_df["question"].tolist()
train_answers   = train_df["answer"].tolist()
test_questions  = test_df["question"].tolist()
test_answers    = test_df["answer"].tolist()

print(f"📊 TRAIN: {len(train_df)} | TEST: {len(test_df)}")

# =========================
# Жолдар
# =========================
drive.mount("/content/drive", force_remount=True)

PROJECT_DIR     = "/content/kz_minilm_st_direct_spm"
LOCAL_ST_DIR    = f"{PROJECT_DIR}/st_finetune_work"   # жұмыс каталог (checkpoint)
ST_OUT_DIR      = f"{PROJECT_DIR}/st_finetuned"       # финетюннен кейінгі финал модель
KK_TOK_DIR      = f"{PROJECT_DIR}/kk_tokenizer"       # қазақ токенизаторы бөлек

Path(PROJECT_DIR).mkdir(parents=True, exist_ok=True)
Path(LOCAL_ST_DIR).mkdir(parents=True, exist_ok=True)
Path(ST_OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(KK_TOK_DIR).mkdir(parents=True, exist_ok=True)

# SPM .model жолын ТЕКСЕРІҢІЗ
spm_model_path = "/content/drive/MyDrive/kazakh_bpe/kazakh_bpe.model"
assert Path(spm_model_path).exists(), f"SPM model not found: {spm_model_path}"

# ============================================================
# 1) SPM .model → Unigram Fast қазақ токенизаторы
# ============================================================
print("🔤 SPM моделін жүктеу және қазақ Unigram токенизаторын дайындау...")
sp = spm.SentencePieceProcessor()
sp.Load(spm_model_path)

# SPM вокабын Unigram Tokenizer форматына айналдыру
vocab = []
get_score = getattr(sp, "GetScore", None)
for i in range(sp.GetPieceSize()):
    token = sp.IdToPiece(i)
    score = float(get_score(i)) if callable(get_score) else 0.0
    vocab.append((token, score))

# UNK id табу
try:
    unk_id = sp.unk_id()
    if unk_id < 0:
        raise ValueError
except Exception:
    unk_id = next(i for i, (t, _) in enumerate(vocab) if t in ("<unk>", "[UNK]"))

# Tokenizers кітапханасының Unigram моделі
tok = Tokenizer(Unigram(vocab=vocab, unk_id=unk_id))

# Metaspace пре/декодер
try:
    tok.pre_tokenizer = PreMS("▁")
except TypeError:
    tok.pre_tokenizer = PreMS()

try:
    tok.decoder = DecMS("▁")
except TypeError:
    tok.decoder = DecMS()

# Арнайы токендер
specials = ["[PAD]", "[CLS]", "[SEP]", "[MASK]"]
tok.add_special_tokens(specials)

# BERT-стиль постпроцессор
tok.post_processor = TemplateProcessing(
    single='[CLS] $A [SEP]',
    pair='[CLS] $A [SEP] $B [SEP]',
    special_tokens=[
        ('[CLS]', tok.token_to_id('[CLS]')),
        ('[SEP]', tok.token_to_id('[SEP]')),
    ],
)

# HF Fast интерфейсі
unk_token_str = "<unk>" if tok.token_to_id("<unk>") is not None else "[UNK]"
kk_tok = PreTrainedTokenizerFast(
    tokenizer_object=tok,
    unk_token=unk_token_str,
    sep_token="[SEP]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    mask_token="[MASK]",
)

print("✅ Қазақ Fast токенизатор дайын. Vocab size:", len(kk_tok))

# Қазақ токенизаторын диске сақтаймыз
kk_tok.save_pretrained(KK_TOK_DIR)
print("📝 Қазақ токенизатор файлдары жазылды →", KK_TOK_DIR)

# ============================================================
# 2) MiniLM encoder + Pooling (қазақ токенизаторымен)
# ============================================================
SENTENCE_T_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print("⬇️ MiniLM encoder-ін жүктеу және қазақ токенизаторымен байланыстыру:", SENTENCE_T_NAME)

word_embedding_model = models.Transformer(
    SENTENCE_T_NAME,
    tokenizer_name_or_path=KK_TOK_DIR
)

# Vocab size-ты қазақ токенизаторына теңестіреміз
word_embedding_model.auto_model.resize_token_embeddings(len(kk_tok))

pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
    pooling_mode_cls_token=False,
    pooling_mode_max_tokens=False,
)

# Толық SentenceTransformer: (ҚАЗАҚ ТОКЕНИЗАТОР → TRANSFORMER → POOLING)
st_model = SentenceTransformer(
    modules=[word_embedding_model, pooling_model],
    device=DEVICE
)

print("✅ Қазақ токенизаторымен біріктірілген ST моделі дайын.")
st_model.save(LOCAL_ST_DIR)
print(f"💾 Бастапқы (pre-finetune) модель сақталды: {LOCAL_ST_DIR}")

# ============================================================
# 3) QA жұптарына fine-tune (MNRL, q→a) – ҚАЗІР ӨШІРІЛГЕН
# ============================================================
BATCH_SIZE = 16
EPOCHS     = 5
WARMUP_PCT = 0.1
LR         = 1e-5

train_examples = [
    InputExample(texts=[row["question"], row["answer"]])
    for _, row in train_df.iterrows()
]

print("\n🧪 Fine-tune конфигурациясы (MNRL, q→a, бірақ бұл нұсқада іске қосылмайды):")
print(f"- TRAIN samples: {len(train_examples)}")
print(f"- BATCH_SIZE: {BATCH_SIZE}")
print(f"- EPOCHS: {EPOCHS}")
print(f"- LR: {LR}")

DO_FINETUNE = False  # <<< fine-tune-ды өшіру флагы

if DO_FINETUNE:
    train_dataloader = DataLoader(
        train_examples,
        shuffle=True,
        batch_size=BATCH_SIZE,
        drop_last=False
    )

    train_loss = losses.MultipleNegativesRankingLoss(st_model)

    num_train_steps = len(train_dataloader) * EPOCHS
    warmup_steps = int(num_train_steps * WARMUP_PCT)

    print(f"- TOTAL STEPS: {num_train_steps}")
    print(f"- WARMUP STEPS: {warmup_steps}")

    print("🚀 Fine-tune басталды (MultipleNegativesRankingLoss, MLM жоқ)...")
    t_ft0 = time.time()
    st_model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=EPOCHS,
        warmup_steps=warmup_steps,
        show_progress_bar=True,
        optimizer_params={"lr": LR},
    )
    print(f"✅ Fine-tune аяқталды (t={time.time()-t_ft0:.2f}s)")
else:
    print("⚠ Fine-tune өшірілген. Pretrained MiniLM + Kazakh tokenizer ғана қолданылады.")

# Fine-tune жасалмаса да, ағымдағы st_model-ді және токенизаторды ST_OUT_DIR-ге сақтаймыз
st_model.save(ST_OUT_DIR)
kk_tok.save_pretrained(f"{ST_OUT_DIR}/0_Transformer")
print(f"💾 ST модель (pretrained, fine-tune ЖОҚ) сақталды: {ST_OUT_DIR}")

# ============================================================
# 4) QA іздеу (held-out TEST, эксперттік бағалау үшін)
# ============================================================

# 🔹 4.1. Retrieval моделі (pretrained/fine-tuned, қазақ токенизаторымен)
retr_model = SentenceTransformer(ST_OUT_DIR, device=DEVICE)
retr_model.to(DEVICE)
retr_model.eval()

def _encode_retr(texts, batch_size=32, normalize=False):
    """Retrieval үшін эмбеддинг (pretrained немесе fine-tuned модель)."""
    with torch.inference_mode():
        vecs = retr_model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=normalize
        )
    return vecs

# ---- Интерактив режим үшін (толық база) ----
qa_questions = qa_data["question"].tolist()
qa_answers   = qa_data["answer"].tolist()
qa_q_emb     = _encode_retr(qa_questions, normalize=True)

# ---- Held-out бағалау үшін TRAIN KB эмбеддингтері ----
train_q_emb = _encode_retr(train_questions, normalize=True)

def ask_question(question, threshold=0.6):
    """
    Интерактив режим: толық QA базасынан іздеу (UX).
    Бұл ғылыми автоматты бағалауға қатыспайды.
    """
    qv = _encode_retr([question], normalize=True)
    sims = (qa_q_emb @ qv.T).squeeze(1)
    idx = int(np.argmax(sims))
    if float(sims[idx]) < threshold:
        return "Кешіріңіз, нақты жауап табылмады.", -1
    return qa_answers[idx], idx

def ask_question_eval(question):
    """
    Held-out бағалау: TEST сұрағы → TRAIN KB.
    threshold қолданбаймыз, әрқашан үздік кандидат.
    Эксперттер pred vs true жауапты ҚОЛМЕН бағалайды.
    """
    qv = _encode_retr([question], normalize=True)
    sims = (train_q_emb @ qv.T).squeeze(1)
    idx = int(np.argmax(sims))
    return train_answers[idx], idx

# 🔁 Диалог (автоматты метрика ЖОҚ)
if __name__ == "__main__":
    try:
        while True:
            user_input = input("\nСұрақ енгізіңіз (шығу үшін 'exit'): ")
            if user_input.strip().lower() == "exit":
                print("Бағдарлама тоқтатылды. 👋")
                break
            answer, idx = ask_question(user_input)
            print("\n=== Жауап ===")
            print(answer)
            print(f"(idx={idx})")
    except EOFError:
        pass




🖥 DEVICE: cuda
📥 QA JSON файлдан жүктеу: qa_data.json
📚 Барлық QA жазба саны: 5944
📊 TRAIN: 5349 | TEST: 595
Mounted at /content/drive
🔤 SPM моделін жүктеу және қазақ Unigram токенизаторын дайындау...
✅ Қазақ Fast токенизатор дайын. Vocab size: 50004
📝 Қазақ токенизатор файлдары жазылды → /content/kz_minilm_st_direct_spm/kk_tokenizer
⬇️ MiniLM encoder-ін жүктеу және қазақ токенизаторымен байланыстыру: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

✅ Қазақ токенизаторымен біріктірілген ST моделі дайын.
💾 Бастапқы (pre-finetune) модель сақталды: /content/kz_minilm_st_direct_spm/st_finetune_work

🧪 Fine-tune конфигурациясы (MNRL, q→a, бірақ бұл нұсқада іске қосылмайды):
- TRAIN samples: 5349
- BATCH_SIZE: 16
- EPOCHS: 5
- LR: 1e-05
⚠ Fine-tune өшірілген. Pretrained MiniLM + Kazakh tokenizer ғана қолданылады.
💾 ST модель (pretrained, fine-tune ЖОҚ) сақталды: /content/kz_minilm_st_direct_spm/st_finetuned

Сұрақ енгізіңіз (шығу үшін 'exit'): Python дегеніміз не?

=== Жауап ===
Python - 1991 жылы Гвидо ван Россум жасаған қарапайым бағдарламалау тілі.
(idx=0)

Сұрақ енгізіңіз (шығу үшін 'exit'): Python бағдарламалау тілінің негізгі мақсаты қандай?

=== Жауап ===
Python бағдарламалау тілінің негізгі мақсаты - ашық бастапқы коды бар көп мақсатты бағдарламалау тілі.
(idx=1)

Сұрақ енгізіңіз (шығу үшін 'exit'): Python - бағдарламалау тілі қандай  салаларда қолданылады?

=== Жауап ===
Python - бағдарламалау тілі көптеген салаларда қолданылад